# US Data Validation Notebook

This notebook validates the 62 Freyberger characteristics for US data.

**Monthly-First Normalization Architecture Output Files:**
- `yearly_raw_characteristics_us.parquet` - Raw yearly characteristics (IDs 6-48, 61)
- `monthly_prices_raw_price_charact_us.parquet` - Raw monthly price characteristics (IDs 1-5, 49-60, 62)
- `final_output_unnormalized_us.parquet` - **Primary validation target** (all 62 raw chars at monthly frequency)
- `final_output_normalized_us.parquet` - All 62 normalized characteristics (monthly cross-sectional rank normalization)

Per Freyberger et al. (2020) and Barroso et al. (2025):
- Annual accounting characteristics are forward-filled to monthly via Fama-French timing
- All 62 characteristics are rank-normalized cross-sectionally **each month**
- Cross-sectional ranks change monthly as the stock universe changes

## 1. Setup and Data Loading


In [8]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add parent directory to path
sys.path.insert(0, str(Path("..").resolve()))

# Import characteristic constants
from data_collection.constants import (
    CHARACTERISTIC_COLUMNS,
    CHARACTERISTIC_METADATA,
    get_characteristics_by_category,
    validate_characteristics_present,
    CATEGORIES,
)

# Configuration
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Data directory
DATA_DIR = Path("../data/outputs")

# File paths (Monthly-First Architecture)
YEARLY_RAW = DATA_DIR / "yearly_raw_characteristics_us.parquet"
FINAL_RAW = DATA_DIR / "final_output_unnormalized_us.parquet"
FINAL_NORM = DATA_DIR / "final_output_normalized_us.parquet"

print("=== Files ===")
print(f"Yearly raw: {YEARLY_RAW.exists()}")
print(f"Final raw (monthly): {FINAL_RAW.exists()}")
print(f"Final norm (monthly): {FINAL_NORM.exists()}")
print(f"\n62 characteristics loaded: {len(CHARACTERISTIC_COLUMNS)}")

=== Files ===
Yearly raw: True
Final raw (monthly): True
Final norm (monthly): True

62 characteristics loaded: 62


In [9]:
# Load final outputs (primary validation)
df_raw = pl.read_parquet(FINAL_RAW) if FINAL_RAW.exists() else None
df_norm = pl.read_parquet(FINAL_NORM) if FINAL_NORM.exists() else None

# Load yearly raw (for detailed checks on accounting characteristics)
df_yearly_raw = pl.read_parquet(YEARLY_RAW) if YEARLY_RAW.exists() else None

if df_raw is not None:
    print(f"Final raw shape: {df_raw.shape}")
    present, missing = validate_characteristics_present(df_raw.columns)
    print(f"\nCharacteristics: {len(present)}/62 present")
    if missing:
        print(f"Missing: {missing}")
    else:
        print("✓ All 62 characteristics found!")
else:
    print("ERROR: Final raw file not found!")

Final raw shape: (975210, 65)

Characteristics: 62/62 present
✓ All 62 characteristics found!


## 2. Schema Validation


In [10]:
# Validate schema
if df_raw is not None:
    print("=== Identifier Columns ===")
    id_cols = ["permno", "gvkey", "date"]
    for col in id_cols:
        if col in df_raw.columns:
            dtype = df_raw.schema[col]
            unique_count = df_raw.select(pl.col(col).n_unique()).item()
            print(f"{col}: {dtype} ({unique_count:,} unique)")

    # Date range
    if "date" in df_raw.columns:
        min_date = df_raw.select(pl.col("date").min()).item()
        max_date = df_raw.select(pl.col("date").max()).item()
        print(f"\nDate range: {min_date} to {max_date}")

    # Characteristics by category
    print("\n=== Characteristics by Category ===")
    for category in CATEGORIES:
        chars_in_cat = get_characteristics_by_category(category)
        present_in_df = [c for c in chars_in_cat if c in df_raw.columns]
        print(f"{category}: {len(present_in_df)}/{len(chars_in_cat)} present")

=== Identifier Columns ===
permno: Int32 (17,099 unique)
gvkey: String (16,857 unique)
date: Date (180 unique)

Date range: 1988-07-29 to 2021-06-30

=== Characteristics by Category ===
Past Returns: 5/5 present
Investment: 6/6 present
Profitability: 17/17 present
Intangibles: 4/4 present
Value: 15/15 present
Trading: 15/15 present


## 3. Null/Missing Analysis


In [11]:
# Null counts for the 62 characteristics
if df_raw is not None:
    print("=== Null Counts by Characteristic ===")

    null_stats = []
    total_rows = len(df_raw)

    for char in CHARACTERISTIC_COLUMNS:
        if char in df_raw.columns:
            null_count = df_raw.select(pl.col(char).null_count()).item()
            null_pct = 100 * null_count / total_rows
            category = CHARACTERISTIC_METADATA[char]["category"]
            null_stats.append(
                {
                    "char": char,
                    "category": category,
                    "null_count": null_count,
                    "null_pct": null_pct,
                }
            )

    # Show top 10 characteristics with most nulls
    null_df = pl.DataFrame(null_stats).sort("null_pct", descending=True)
    print("\nTop 10 characteristics with most nulls:")
    print(null_df.head(10))

    # Characteristics with >50% nulls
    high_null = null_df.filter(pl.col("null_pct") > 50)
    if len(high_null) > 0:
        print(f"\n⚠️ {len(high_null)} characteristics have >50% nulls:")
        print(high_null)
    else:
        print("\n✓ No characteristics with >50% nulls")

=== Null Counts by Characteristic ===

Top 10 characteristics with most nulls:
shape: (10, 4)
┌──────────┬───────────────┬────────────┬───────────┐
│ char     ┆ category      ┆ null_count ┆ null_pct  │
│ ---      ┆ ---           ┆ ---        ┆ ---       │
│ str      ┆ str           ┆ i64        ┆ f64       │
╞══════════╪═══════════════╪════════════╪═══════════╡
│ AOA      ┆ Intangibles   ┆ 334310     ┆ 34.280822 │
│ OA       ┆ Intangibles   ┆ 334310     ┆ 34.280822 │
│ Free_CF  ┆ Value         ┆ 326400     ┆ 33.469714 │
│ Beta_Cor ┆ Trading       ┆ 304308     ┆ 31.204356 │
│ Beta     ┆ Trading       ┆ 304308     ┆ 31.204356 │
│ Idio_vol ┆ Trading       ┆ 304308     ┆ 31.204356 │
│ r36_13   ┆ Past Returns  ┆ 255232     ┆ 26.172004 │
│ ROE      ┆ Profitability ┆ 204734     ┆ 20.993837 │
│ DPI2A    ┆ Investment    ┆ 201987     ┆ 20.712154 │
│ O2P      ┆ Value         ┆ 183655     ┆ 18.832354 │
└──────────┴───────────────┴────────────┴───────────┘

✓ No characteristics with >50% nulls


## 4. Value Distribution Analysis


In [12]:
# Summary statistics for characteristics
if df_raw is not None:
    print("=== Summary Statistics by Category ===")

    for category in CATEGORIES:
        chars = get_characteristics_by_category(category)
        present_chars = [c for c in chars if c in df_raw.columns]

        if present_chars:
            print(f"\n{category}:")
            stats = df_raw.select(present_chars).describe()
            print(stats)

=== Summary Statistics by Category ===

Past Returns:
shape: (9, 6)
┌────────────┬───────────┬───────────┬────────────┬───────────┬───────────┐
│ statistic  ┆ r2_1      ┆ r6_2      ┆ r12_2      ┆ r12_7     ┆ r36_13    │
│ ---        ┆ ---       ┆ ---       ┆ ---        ┆ ---       ┆ ---       │
│ str        ┆ f64       ┆ f64       ┆ f64        ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═══════════╪════════════╪═══════════╪═══════════╡
│ count      ┆ 966979.0  ┆ 943987.0  ┆ 914203.0   ┆ 915897.0  ┆ 719978.0  │
│ null_count ┆ 8231.0    ┆ 31223.0   ┆ 61007.0    ┆ 59313.0   ┆ 255232.0  │
│ mean       ┆ 0.014712  ┆ 0.073751  ┆ 0.148776   ┆ 0.068182  ┆ 0.283273  │
│ std        ┆ 0.184291  ┆ 0.459668  ┆ 0.812598   ┆ 0.485161  ┆ 1.147334  │
│ min        ┆ -0.9936   ┆ -0.999974 ┆ -0.999995  ┆ -0.999971 ┆ -1.0      │
│ 25%        ┆ -0.059964 ┆ -0.130417 ┆ -0.197131  ┆ -0.156553 ┆ -0.266668 │
│ 50%        ┆ 0.00147   ┆ 0.029423  ┆ 0.054193   ┆ 0.023188  ┆ 0.124998  │
│ 75%        ┆ 0.070

## 5. Normalized Range Validation


In [13]:
# Check normalized characteristics are in expected range
if df_norm is not None:
    print("=== Normalized Range Check ===")
    print("Expected range: approximately [-0.5, 0.5]")
    print("(Monthly cross-sectional rank normalization per Barroso et al. 2025)\n")

    # Check each normalized characteristic (with _norm suffix)
    norm_chars = [f"{c}_norm" for c in CHARACTERISTIC_COLUMNS]
    present_norm = [c for c in norm_chars if c in df_norm.columns]
    missing_norm = [c for c in norm_chars if c not in df_norm.columns]

    print(f"Normalized characteristics found: {len(present_norm)}/62")
    if missing_norm:
        print(f"Missing normalized: {missing_norm[:10]}...")

    out_of_range = []
    for char in present_norm:
        min_val = df_norm.select(pl.col(char).min()).item()
        max_val = df_norm.select(pl.col(char).max()).item()

        # Check if within reasonable bounds (allowing some flexibility)
        if min_val is not None and max_val is not None:
            if min_val < -0.6 or max_val > 0.6:
                out_of_range.append({"char": char, "min": min_val, "max": max_val})

    if out_of_range:
        print(f"\n⚠️ {len(out_of_range)} characteristics outside expected range:")
        for item in out_of_range[:10]:  # Show first 10
            print(f"  {item['char']}: [{item['min']:.3f}, {item['max']:.3f}]")
    else:
        print("\n✓ All normalized characteristics within expected range")

    # === Key validation: monthly variation in accounting characteristics ===
    # In the new pipeline, accounting chars should have different normalized values
    # each month, even within the same fiscal year (because cross-sectional ranks change)
    print("\n=== Monthly Variation Check (Accounting Characteristics) ===")
    print("Accounting chars should vary month-to-month even within a fiscal year.")

    if "date" in df_norm.columns and "BEME_norm" in df_norm.columns:
        # Pick a sample year and check if BEME_norm varies by month
        sample = df_norm.filter(pl.col("date").dt.year() == 2015)
        if len(sample) > 0:
            monthly_means = (
                sample.group_by("date")
                .agg(pl.col("BEME_norm").mean().alias("mean_BEME_norm"))
                .sort("date")
            )
            n_months = monthly_means.shape[0]
            n_unique = monthly_means["mean_BEME_norm"].n_unique()
            print(
                f"  BEME_norm in 2015: {n_months} months, {n_unique} unique mean values"
            )
            if n_unique > 1:
                print("  ✓ Monthly variation confirmed (cross-sectional ranks change)")
            else:
                print("  ⚠️ No monthly variation detected - check normalization")
        else:
            print("  (No data for 2015 sample year)")
    else:
        print("  (Cannot check - missing date or BEME_norm column)")
else:
    print("Normalized file not available for range check")

=== Normalized Range Check ===
Expected range: approximately [-0.5, 0.5]
(Monthly cross-sectional rank normalization per Barroso et al. 2025)

Normalized characteristics found: 62/62

✓ All normalized characteristics within expected range

=== Monthly Variation Check (Accounting Characteristics) ===
Accounting chars should vary month-to-month even within a fiscal year.
  BEME_norm in 2015: 6 months, 6 unique mean values
  ✓ Monthly variation confirmed (cross-sectional ranks change)


## 6. Characteristic Metadata Reference


In [14]:
# Display characteristic metadata
print("=== 62 Freyberger Characteristics ===")
print("\nOrganized by category:\n")

for category in CATEGORIES:
    chars = get_characteristics_by_category(category)
    print(f"\n{category} ({len(chars)} characteristics):")
    print("-" * 60)

    for char in chars:
        meta = CHARACTERISTIC_METADATA[char]
        print(f"  [{meta['id']:2d}] {char:12s} - {meta['description']}")
        print(f"       Formula: {meta['formula']}")

=== 62 Freyberger Characteristics ===

Organized by category:


Past Returns (5 characteristics):
------------------------------------------------------------
  [ 1] r2_1         - Return 1 month before prediction
       Formula: ret(t-1)
  [ 2] r6_2         - Return 6 to 2 months before
       Formula: cumret(t-6, t-2)
  [ 3] r12_2        - Return 12 to 2 months before
       Formula: cumret(t-12, t-2)
  [ 4] r12_7        - Return 12 to 7 months before
       Formula: cumret(t-12, t-7)
  [ 5] r36_13       - Return 36 to 13 months before
       Formula: cumret(t-36, t-13)

Investment (6 characteristics):
------------------------------------------------------------
  [ 6] Investment   - % change in AT
       Formula: Δat / at_{t-1}
  [ 7] ACEQ         - % change in Book Equity
       Formula: Δceq / ceq_{t-1}
  [ 8] DPI2A        - Change in PP&E + inventory / lagged AT
       Formula: (Δppent + Δinvt) / at_{t-1}
  [ 9] AShrout      - % change in shares outstanding
       Formula: Δshare